# measurecv on a GPU (Colab / Kaggle)

Metric object measurement from a single RGB photo:
**RT-DETR** detection -> **SAM 2** segmentation -> **Metric3D** metric depth -> calibrated geometry.

### There is nothing to train

All three models are pretrained foundation models used **zero-shot**. The GPU here is for *running* the large weights, not fine-tuning. You would only train if you needed a custom object class that RT-DETR's 80 COCO classes do not cover — and even then, only the detector.

### Before you start

Enable the GPU: **Runtime -> Change runtime type -> T4 GPU** (Colab), or **Settings -> Accelerator -> GPU** (Kaggle).

In [ ]:
# 1. Confirm the GPU is actually attached before downloading 3 GB of weights.
import subprocess, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)

import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}  {p.total_memory/1e9:.1f} GB  compute {p.major}.{p.minor}")
    print(f"bf16 supported: {p.major >= 8}")
else:
    print("\n!! No GPU. Enable it in the runtime settings, or use configs/cpu.yaml.")

In [ ]:
# 2. Install.
#
# `timm` and `mmengine` are genuine runtime dependencies of Metric3D's hub
# script that it does not declare. `mmengine` is pure Python -- you do NOT
# need OpenMMLab's compiled `mmcv`, which has no wheels for recent Python.
#
# subprocess rather than ! magics: identical behaviour in Colab and Kaggle,
# and errors actually surface instead of scrolling past.

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = ""  # e.g. "https://github.com/you/measurecv.git" -- blank to upload a zip


def run(cmd):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError(f"failed: {' '.join(cmd)}")
    return result.stdout


if not Path("pyproject.toml").exists():
    if REPO_URL:
        run(["git", "clone", "--depth", "1", REPO_URL, "measurecv_repo"])
        os.chdir("measurecv_repo")
    else:
        try:
            from google.colab import files

            print("Upload a zip of the project:")
            uploaded = files.upload()
            name = next(iter(uploaded))
            run(["unzip", "-oq", name])
            # Enter the directory that actually contains the project.
            for candidate in Path(".").rglob("pyproject.toml"):
                os.chdir(candidate.parent)
                break
        except ImportError:
            raise SystemExit(
                "Not in the project directory. Set REPO_URL, or upload and unzip it."
            ) from None

print("working dir:", Path.cwd())
assert Path("pyproject.toml").exists(), "pyproject.toml not found"

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[models,api]", "timm", "mmengine"])
print("\ninstalled")

In [ ]:
# 3. Download the weights (~2.9 GB, cached for the rest of the session).
#
#   RT-DETRv2 r101vd       307 MB   strongest detector backbone
#   SAM 2.1 hiera-large    898 MB   best mask boundaries
#   Metric3D ViT-Large    1648 MB   metric depth
#
# Swap `configs/gpu.yaml` for `configs/high_accuracy.yaml` to use
# Metric3D ViT-giant2 instead (5.5 GB, ~12 GB VRAM in fp32).

import time
from measurecv import load_config
from measurecv.pipeline.pipeline import MeasurementPipeline

cfg = load_config("configs/gpu.yaml")
pipeline = MeasurementPipeline(cfg)

t0 = time.time()
pipeline.models.load_all()   # downloads on first call
pipeline.warmup()            # pays the cuDNN autotune cost once
print(f"models ready in {time.time()-t0:.1f}s")
print(pipeline.models.info())

In [ ]:
# 4. Get a photo. Upload your own, or fall back to a COCO sample.
from pathlib import Path
import urllib.request

IMAGE = None
try:
    from google.colab import files
    print("Upload a photo (or press Cancel to use a sample):")
    up = files.upload()
    if up:
        IMAGE = next(iter(up))
except Exception:
    pass

if not IMAGE:
    IMAGE = "sample.jpg"
    urllib.request.urlretrieve(
        "http://images.cocodataset.org/val2017/000000000139.jpg", IMAGE)
    print("using COCO sample")

print(f"image: {IMAGE}  ({Path(IMAGE).stat().st_size/1024:.0f} KB)")

In [ ]:
# 5. Measure.
import time

from measurecv import Frame
from measurecv.pipeline.sources import read_image

t0 = time.time()
artifacts = pipeline.measure_frame_full(Frame(image=read_image(IMAGE)), track=False)
scene = artifacts.scene
print(f"measured in {time.time()-t0:.2f}s\n")

print(f"{'label':<16}{'L x W x H (m)':<28}{'volume':>10}{'dist':>9}{'conf':>7}")
print("-" * 70)
for o in scene.objects:
    if o.dimensions is None:
        print(f"{o.detection.label:<16}{'not measurable':<28}")
        continue
    d = o.dimensions
    dims = f"{d.length.value:.3f} x {d.width.value:.3f} x {d.height.value:.3f}"
    vol = f"{o.volume.value*1000:.1f} L" if o.volume else "-"
    dist = f"{o.distance.value:.2f} m" if o.distance else "-"
    print(f"{o.detection.label:<16}{dims:<28}{vol:>10}{dist:>9}{o.confidence:>6.0%}")

print(f"\ncalibration: {scene.calibration_source}")
print(f"timings ms:  { {k: round(v) for k, v in scene.timings_ms.items()} }")

In [ ]:
# 6. Look at it.
import matplotlib.pyplot as plt
from measurecv.viz.annotate import AnnotationStyle, draw_depth_map, draw_scene

annotated = draw_scene(
    artifacts.image, scene,
    masks=[m.mask for m in artifacts.masks],
    intrinsics=artifacts.intrinsics,
    style=AnnotationStyle(show_volume=True, max_labels=8),
)
depth_vis = draw_depth_map(artifacts.depth_map.depth)

fig, ax = plt.subplots(1, 2, figsize=(22, 9))
ax[0].imshow(annotated); ax[0].set_title("measurements"); ax[0].axis("off")
ax[1].imshow(depth_vis); ax[1].set_title("metric depth"); ax[1].axis("off")
plt.tight_layout(); plt.show()

print("depth range:", artifacts.depth_map.stats())

## Getting accurate numbers

Everything above ran **uncalibrated**, so absolute scale carries roughly **15%** uncertainty — check `calibration_source: assumed_fov` in the output. Two ways to fix that, in order of impact.

In [ ]:
# 7a. Calibrate the camera -- the biggest single improvement (~15% -> ~1-2%).
#
# Print a chessboard, photograph it 10-15 times from varied angles with the
# SAME camera and settings you use for measuring, then upload them here.
#
# Measure the printed square with a ruler. Printers scale, and every
# measurement this system produces is directly proportional to that number.

from measurecv.calibration.board import calibrate_from_paths

BOARD = (9, 6)          # INNER corners, not squares
SQUARE_M = 0.025        # measure it!

paths = sorted(Path("calib").glob("*.jpg")) if Path("calib").exists() else []
if paths:
    result = calibrate_from_paths(
        paths, board_shape=BOARD, square_size_m=SQUARE_M, min_views=8)
    result.intrinsics.save("camera.json")
    print(f"fx={result.intrinsics.fx:.1f}  RMS={result.rms_error:.3f}px  "
          f"coverage={result.coverage:.0%}")
    pipeline.resolver.set_profile(result.intrinsics)
    print("profile active -- re-run cell 5")
else:
    print("No images in ./calib -- upload chessboard photos there first.")

In [ ]:
# 7b. Or put a known object in the shot and cancel the residual bias.
#
# Measure something whose true size you know (a credit card is 85.60 mm by
# ISO/IEC 7810, A4 is 297 mm), then feed both numbers back.

from measurecv.calibration.scale import estimate_scale_correction

MEASURED_M = [0.092]   # what the table above reported
TRUE_M     = [0.0856]  # credit card long edge

correction = estimate_scale_correction(MEASURED_M, TRUE_M, reference="credit_card")
pipeline.set_scale_correction(correction)
print(correction.to_dict())
print("\nApplied. Re-run cell 5 -- lengths scale by the factor, areas by its")
print("square and volumes by its cube, and the error bars tighten accordingly.")

In [ ]:
# 8. Export.
from measurecv.export.serializers import write_csv, write_json
from measurecv.geometry.backproject import backproject_depth_map
from measurecv.viz.export3d import write_ply

write_json("result.json", scene)
write_csv("result.csv", [scene])

# The point cloud is the fastest way to sanity-check a suspicious measurement:
# depth bleed and mask leakage are obvious in 3-D and invisible in JSON.
cloud = backproject_depth_map(
    artifacts.depth_map, artifacts.intrinsics, stride=2, image=artifacts.image)
write_ply("scene.ply", cloud)
print(f"wrote result.json, result.csv, scene.ply ({len(cloud):,} points)")

try:
    from google.colab import files
    files.download("result.json")
except ImportError:
    pass

## Benchmarking and video

```python
# per-stage latency -- tells you which model to shrink if you need speed
!measurecv benchmark -c configs/gpu.yaml -n 20 --warmup 3

# video with tracking and temporal fusion
!measurecv video clip.mp4 -c configs/gpu.yaml \
     --output results.ndjson --csv results.csv --render annotated.mp4
```

## If you hit an out-of-memory error

In order of what to try first:

1. `runtime.precision: auto` — bf16/fp16 roughly halves the weight memory.
2. `runtime.max_image_side: 1024` — activations scale with pixel count.
3. Use `configs/gpu.yaml` rather than `high_accuracy.yaml` (ViT-Large instead of the 5.5 GB ViT-giant2).
4. `segmentation.model_id: facebook/sam2.1-hiera-base-plus`.

## Reading the output

Every quantity is `{value, sigma, unit, confidence, interval_95}`:

- **`sigma`** — physical 1-sigma error bar, given the method applies.
- **`confidence`** — whether the method applies at all (mask quality, truncation, conditioning, calibration provenance).

A truncated object is capped at 0.4 confidence because its measurements are lower bounds, not measurements. See `docs/accuracy.md` for the full error budget.